# CSV 데이터로 의사결정트리 분류하기

이 노트북은 구글 코랩에서 CSV 파일을 업로드한 뒤 데이터를 확인하고, 사용자가 선택한 종속변수와 독립변수로 분류 의사결정트리를 생성합니다.

- CSV 인코딩은 `utf-8-sig`, `utf-8`, `cp949`, `euc-kr` 순서로 시도합니다.
- 종속변수는 드롭다운으로 선택합니다.
- 독립변수는 체크박스로 선택합니다.
- 트리 제약 조건은 고급 설정을 켰을 때만 조정합니다.
- 트리 그림은 PNG와 PDF로 저장하고, 리프 노드 규칙표는 CSV와 HTML로 저장합니다.

## 1. 라이브러리 준비

한글 그래프 출력을 위해 `koreanize-matplotlib`을 사용합니다.

In [ ]:
# @title
!pip -q install koreanize-matplotlib

import html
import io
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

try:
    from google.colab import files
except Exception:
    files = None

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = False

print("준비 완료")



## 2. 데이터 기본 정보 확인

In [ ]:
# @title
df = pd.read_csv("https://raw.githubusercontent.com/CarlosQuperman/AIEDAP2026_LOCAL_DATA/refs/heads/main/5%EC%9E%A5/%EC%B6%98%EC%B2%9C_%EB%8B%AD%EA%B0%88%EB%B9%84_%EC%83%81%EC%A0%90%EB%B3%84_%EC%B5%9C%EA%B3%A0_%EC%9D%B4%EC%9A%A9%EC%9E%90%EC%88%98(%EC%83%81%EC%9C%8415).csv")

def show_df_info(dataframe):
    buffer = io.StringIO()
    dataframe.info(buf=buffer)
    display(HTML("<h4>df.info()</h4>"))
    display(HTML(f"<pre>{buffer.getvalue()}</pre>"))


def column_profile(dataframe):
    rows = []
    total = len(dataframe)

    for col in dataframe.columns:
        missing = int(dataframe[col].isna().sum())
        examples = dataframe[col].dropna().astype(str).unique()[:8]
        rows.append({
            "컬럼명": col,
            "자료형": str(dataframe[col].dtype),
            "결측치": missing,
            "결측비율(%)": round(missing / total * 100, 2) if total else 0,
            "고유값 수": int(dataframe[col].nunique(dropna=True)),
            "값 예시": ", ".join(examples),
        })

    return pd.DataFrame(rows)


show_df_info(df)
display(HTML("<h4>컬럼별 결측치와 고유값</h4>"))
display(column_profile(df))

numeric_df = df.select_dtypes(include=[np.number])
if not numeric_df.empty:
    display(HTML("<h4>숫자형 컬럼 describe()</h4>"))
    display(numeric_df.describe().T)
else:
    display(HTML("숫자형 컬럼이 없습니다."))


## 3. 의사결정나무 분류 작업을 위한 도구 정의하기


In [ ]:
# @title
#분류 트리 전처리 함수

def normalize_dataframe(dataframe):
    cleaned = dataframe.copy()

    for col in cleaned.columns:
        if pd.api.types.is_object_dtype(cleaned[col]) or pd.api.types.is_string_dtype(cleaned[col]):
            cleaned[col] = cleaned[col].map(lambda x: x.strip() if isinstance(x, str) else x)
            cleaned[col] = cleaned[col].replace(r"^\s*$", np.nan, regex=True)

    return cleaned


def conversion_ratio(series, converter):
    non_missing = series.dropna()
    if len(non_missing) == 0:
        return 0
    converted = converter(non_missing)
    return converted.notna().mean()


def date_success_ratio(series):
    non_missing = series.dropna()
    if len(non_missing) == 0:
        return 0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        converted = pd.to_datetime(non_missing, errors="coerce")
    return converted.notna().mean()


def prepare_classification_data(dataframe, feature_cols, target_col):
    cleaned = normalize_dataframe(dataframe)
    messages = []
    warning_messages = []

    before = len(cleaned)
    cleaned = cleaned.dropna(subset=[target_col]).copy()
    if len(cleaned) < before:
        messages.append(f"종속변수 '{target_col}'가 비어 있는 {before - len(cleaned)}행을 제외했습니다.")

    y = cleaned[target_col].astype(str)
    if y.nunique() < 2:
        raise ValueError("종속변수의 고유값이 2개 미만이라 분류 트리를 만들 수 없습니다.")

    if y.nunique() > 30 or y.nunique() / max(len(y), 1) > 0.3:
        warning_messages.append(
            f"종속변수 '{target_col}'의 고유값이 {y.nunique()}개입니다. 고유값이 많은 컬럼은 과적합될 수 있습니다."
        )

    numeric_parts = []
    categorical_parts = []

    for col in feature_cols:
        series = cleaned[col]
        unique_count = series.dropna().nunique()

        if unique_count <= 1:
            warning_messages.append(f"독립변수 '{col}'은 고유값이 1개 이하라 분기에 거의 기여하지 않을 수 있습니다.")

        numeric_ratio = conversion_ratio(series, lambda s: pd.to_numeric(s, errors="coerce"))
        date_ratio = date_success_ratio(series)

        if pd.api.types.is_numeric_dtype(series) or numeric_ratio >= 0.95:
            numeric_series = pd.to_numeric(series, errors="coerce")
            fill_value = numeric_series.median()
            if pd.isna(fill_value):
                fill_value = 0
            numeric_parts.append(numeric_series.fillna(fill_value).rename(col))
            messages.append(f"'{col}': 숫자형으로 사용, 결측은 중앙값({fill_value})으로 대체")
        elif date_ratio >= 0.8 and unique_count > 2:
            dt = pd.to_datetime(series, errors="coerce")
            numeric_parts.extend([
                dt.dt.year.fillna(0).astype(int).rename(f"{col}_연"),
                dt.dt.month.fillna(0).astype(int).rename(f"{col}_월"),
                dt.dt.day.fillna(0).astype(int).rename(f"{col}_일"),
                dt.dt.dayofweek.fillna(0).astype(int).rename(f"{col}_요일"),
            ])
            messages.append(f"'{col}': 날짜형으로 해석해 연/월/일/요일 변수로 변환")
        else:
            cat = series.fillna("결측").astype(str).replace(r"^\s*$", "결측", regex=True)
            categorical_parts.append(cat.rename(col))
            messages.append(f"'{col}': 명목형으로 사용, 결측은 '결측' 범주로 대체 후 원-핫 인코딩")
            if unique_count > 30:
                warning_messages.append(f"독립변수 '{col}'의 고유값이 {unique_count}개라 원-핫 인코딩 후 변수가 많아질 수 있습니다.")

    parts = []
    if numeric_parts:
        parts.append(pd.concat(numeric_parts, axis=1))
    if categorical_parts:
        cat_df = pd.concat(categorical_parts, axis=1)
        parts.append(pd.get_dummies(cat_df, prefix=cat_df.columns, prefix_sep=" = ", dtype=int))

    if not parts:
        raise ValueError("사용 가능한 독립변수가 없습니다.")

    X = pd.concat(parts, axis=1)
    X = X.loc[:, X.nunique(dropna=False) > 1]
    if X.empty:
        raise ValueError("전처리 후 남은 독립변수가 없습니다.")

    return X, y, messages, warning_messages


#트리 생성과 저장 함수

def train_tree(dataframe, feature_cols, target_col, max_depth=None, min_samples_leaf=1):
    X, y, messages, warning_messages = prepare_classification_data(dataframe, feature_cols, target_col)

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
    )
    model.fit(X, y)

    pred = model.predict(X)
    metrics = {
        "전체 데이터 행 수": len(X),
        "전처리 후 독립변수 수": X.shape[1],
        "종속변수 고유값 수": y.nunique(),
        "트리 깊이": model.get_depth(),
        "리프 노드 수": model.get_n_leaves(),
        "전체 데이터 기준 정확도": round(accuracy_score(y, pred), 4),
    }

    return model, X, y, metrics, messages, warning_messages


def safe_stem(text):
    stem = re.sub(r"[^0-9A-Za-z가-힣_ -]+", "_", Path(str(text)).stem).strip()
    stem = re.sub(r"\s+", "_", stem)
    return stem or "decision_tree_result"


def readable_condition(feature_name, threshold, direction):
    """Convert sklearn threshold text into a beginner-friendly condition."""
    if " = " in feature_name and abs(threshold - 0.5) < 1e-6:
        column, value = feature_name.split(" = ", 1)
        if direction == "right":
            return f"{column} = {value}", "범주형 값을 원-핫 인코딩한 조건입니다. 이 가지는 해당 값에 속한다는 뜻입니다."
        return f"{column} ≠ {value}", "범주형 값을 원-핫 인코딩한 조건입니다. 이 가지는 해당 값에 속하지 않는다는 뜻입니다."

    operator = "≤" if direction == "left" else ">"
    return f"{feature_name} {operator} {threshold:.3f}", "숫자형 조건입니다. 값이 기준보다 작은지 또는 큰지를 기준으로 나눕니다."


def node_prediction(model, node_id):
    values = model.tree_.value[node_id][0]
    return str(model.classes_[int(np.argmax(values))])


def class_distribution_text(model, node_id, max_items=8):
    values = model.tree_.value[node_id][0]
    pairs = []
    for label, count in zip(model.classes_, values):
        if count > 0:
            pairs.append((str(label), int(count)))
    pairs = sorted(pairs, key=lambda x: x[1], reverse=True)[:max_items]
    return ", ".join([f"{label}: {count}" for label, count in pairs])


def make_interactive_tree_html(model, feature_names, output_path):
    tree = model.tree_
    total_nodes = tree.node_count
    max_depth = model.get_depth()
    leaf_count = model.get_n_leaves()
    internal_count = total_nodes - leaf_count

    level_counts = {}
    leaf_samples = []
    leaf_ginis = []
    question_scores = []

    def collect_summary(node_id, level):
        level_counts[level] = level_counts.get(level, 0) + 1
        left = tree.children_left[node_id]
        right = tree.children_right[node_id]

        if left == right:
            leaf_samples.append(int(tree.n_node_samples[node_id]))
            leaf_ginis.append(float(tree.impurity[node_id]))
            return

        parent_samples = tree.n_node_samples[node_id]
        left_weight = tree.n_node_samples[left] / parent_samples
        right_weight = tree.n_node_samples[right] / parent_samples
        weighted_child_gini = left_weight * tree.impurity[left] + right_weight * tree.impurity[right]
        gini_gain = tree.impurity[node_id] - weighted_child_gini
        feature_name = feature_names[tree.feature[node_id]]
        threshold = tree.threshold[node_id]
        if " = " in feature_name and abs(threshold - 0.5) < 1e-6:
            column, value = feature_name.split(" = ", 1)
            question = f"{column}이(가) {value}인가?"
        else:
            question = f"{feature_name} ≤ {threshold:.3f}인가?"
        question_scores.append({
            "노드": node_id,
            "질문": question,
            "gini 감소량": round(float(gini_gain), 4),
            "샘플 수": int(parent_samples),
        })
        collect_summary(left, level + 1)
        collect_summary(right, level + 1)

    collect_summary(0, 0)
    level_summary = ", ".join([f"레벨 {level}: {count}개" for level, count in sorted(level_counts.items())])
    top_questions = sorted(question_scores, key=lambda item: item["gini 감소량"], reverse=True)[:5]
    top_question_rows = "".join([
        f"<tr><td>{idx}</td><td>{html.escape(item['질문'])}</td><td>{item['gini 감소량']}</td><td>{item['샘플 수']}</td></tr>"
        for idx, item in enumerate(top_questions, start=1)
    ]) or "<tr><td colspan='4'>분기 질문이 없습니다.</td></tr>"
    tree_summary_html = f"""
    <div class='tree-summary'>
      <h2>트리 요약</h2>
      <div class='summary-grid'>
        <div><b>전체 노드 수</b><br>{total_nodes}</div>
        <div><b>분기 노드 수</b><br>{internal_count}</div>
        <div><b>리프 노드 수</b><br>{leaf_count}</div>
        <div><b>최대 계층</b><br>{max_depth}</div>
        <div><b>리프 최소 샘플</b><br>{min(leaf_samples) if leaf_samples else 0}</div>
        <div><b>리프 최대 샘플</b><br>{max(leaf_samples) if leaf_samples else 0}</div>
        <div><b>리프 평균 gini</b><br>{round(float(np.mean(leaf_ginis)), 4) if leaf_ginis else 0}</div>
      </div>
      <p><b>계층별 노드 수:</b> {html.escape(level_summary)}</p>
      <h3>노드를 잘 나눈 상위 5개 질문</h3>
      <p class='small-help'>gini 감소량이 클수록 해당 질문이 데이터를 더 깔끔하게 나누었다는 뜻입니다.</p>
      <table class='question-table'><thead><tr><th>순위</th><th>질문</th><th>gini 감소량</th><th>샘플 수</th></tr></thead><tbody>{top_question_rows}</tbody></table>
    </div>
    """

    def build(node_id, level, branch_label="시작", branch_help="루트 노드입니다."):
        left = tree.children_left[node_id]
        right = tree.children_right[node_id]
        is_leaf = left == right
        samples = int(tree.n_node_samples[node_id])
        impurity = float(tree.impurity[node_id])
        prediction = html.escape(node_prediction(model, node_id))
        distribution = html.escape(class_distribution_text(model, node_id))
        branch_label_html = html.escape(branch_label)
        branch_help_html = html.escape(branch_help)

        if is_leaf:
            node_title = f"리프 노드 {node_id}: {prediction}"
            test_text = "더 이상 나누지 않고 이 노드의 예측값을 사용합니다."
            children_html = ""
            node_class = "leaf"
        else:
            feature_name = feature_names[tree.feature[node_id]]
            threshold = tree.threshold[node_id]
            left_text, left_help = readable_condition(feature_name, threshold, "left")
            right_text, right_help = readable_condition(feature_name, threshold, "right")
            node_title = f"분기 노드 {node_id}"
            test_text = f"이 노드는 '{html.escape(feature_name)}' 값을 기준으로 두 갈래로 나눕니다."
            children_html = (
                "<div class='children'>"
                + build(left, level + 1, left_text, left_help)
                + build(right, level + 1, right_text, right_help)
                + "</div>"
            )
            node_class = "internal"

        open_attr = " open" if level < 2 else ""
        return f"""
        <details class='node {node_class}'{open_attr}>
          <summary>
            <span class='branch' title='{branch_help_html}'>{branch_label_html}</span>
            <span class='title'>{html.escape(node_title)}</span>
            <span class='badge'>레벨 {level}</span>
            <span class='badge'>샘플 {samples}</span>
            <span class='badge' title='gini는 한 노드 안에 여러 분류가 섞여 있는 정도입니다. 0이면 한 종류만 있어 매우 순수합니다.'>gini {impurity:.3f}</span>
          </summary>
          <div class='body'>
            <div><b>예측값:</b> {prediction}</div>
            <div><b>노드 설명:</b> {test_text}</div>
            <div><b>분류 분포:</b> {distribution if distribution else '표시할 분포 없음'}</div>
          </div>
          {children_html}
        </details>
        """

    tree_html = build(0, 0)

    graph_nodes = []
    graph_edges = []
    leaf_index = 0
    h_gap = 260
    v_gap = 155
    margin_x = 100
    margin_y = 70
    node_w = 210
    node_h = 90

    def short_text(text, limit=28):
        text = str(text)
        return text if len(text) <= limit else text[:limit - 1] + "…"

    def svg_lines(text, x, y, max_chars=17, max_lines=3, line_gap=17, css_class=""):
        text = str(text)
        chunks = [text[i:i + max_chars] for i in range(0, len(text), max_chars)] or [""]
        chunks = chunks[:max_lines]
        if len(text) > max_chars * max_lines:
            chunks[-1] = chunks[-1][:-1] + "…"
        return "".join([
            f"<text class='{css_class}' x='{x}' y='{y + idx * line_gap}' text-anchor='middle'>{html.escape(chunk)}</text>"
            for idx, chunk in enumerate(chunks)
        ])

    def layout_graph(node_id, level, branch_label="시작", branch_help="루트 노드입니다.", path=None):
        nonlocal leaf_index
        if path is None:
            path = []
        left = tree.children_left[node_id]
        right = tree.children_right[node_id]
        is_leaf = left == right
        y = margin_y + level * v_gap

        if is_leaf:
            x = margin_x + leaf_index * h_gap
            leaf_index += 1
            node_kind = "leaf"
        else:
            feature_name = feature_names[tree.feature[node_id]]
            threshold = tree.threshold[node_id]
            left_text, left_help = readable_condition(feature_name, threshold, "left")
            right_text, right_help = readable_condition(feature_name, threshold, "right")
            left_x, left_y = layout_graph(left, level + 1, left_text, left_help, path + [left_text])
            right_x, right_y = layout_graph(right, level + 1, right_text, right_help, path + [right_text])
            x = (left_x + right_x) / 2
            graph_edges.append({"source": (x, y), "target": (left_x, left_y), "label": left_text})
            graph_edges.append({"source": (x, y), "target": (right_x, right_y), "label": right_text})
            node_kind = "internal"

        prediction = node_prediction(model, node_id)
        distribution = class_distribution_text(model, node_id)
        graph_nodes.append({
            "id": node_id,
            "x": x,
            "y": y,
            "level": level,
            "kind": node_kind,
            "branch": branch_label,
            "branch_help": branch_help,
            "path": path,
            "prediction": prediction,
            "samples": int(tree.n_node_samples[node_id]),
            "gini": float(tree.impurity[node_id]),
            "distribution": distribution,
        })
        return x, y

    layout_graph(0, 0)
    svg_width = max([node["x"] for node in graph_nodes] + [900]) + margin_x
    svg_height = max([node["y"] for node in graph_nodes] + [500]) + margin_y

    edge_svg = ""
    for edge in graph_edges:
        sx, sy = edge["source"]
        tx, ty = edge["target"]
        mx = (sx + tx) / 2
        my = (sy + ty) / 2 - 8
        edge_svg += f"<line class='edge' x1='{sx}' y1='{sy + node_h / 2}' x2='{tx}' y2='{ty - node_h / 2}' />"
        edge_svg += f"<text class='edge-label' x='{mx}' y='{my}' text-anchor='middle'>{html.escape(short_text(edge['label'], 24))}</text>"

    node_svg = ""
    for node in graph_nodes:
        x = node["x"]
        y = node["y"]
        rect_x = x - node_w / 2
        rect_y = y - node_h / 2
        title = f"노드 {node['id']} | {node['branch']} | 예측값: {node['prediction']} | 샘플: {node['samples']} | gini: {node['gini']:.3f}"
        path_text = "||".join(node["path"])
        node_svg += f"""
        <g class='svg-node {node['kind']}' tabindex='0'
           data-node='{node['id']}'
           data-level='{node['level']}'
           data-branch='{html.escape(node['branch'], quote=True)}'
           data-help='{html.escape(node['branch_help'], quote=True)}'
           data-path='{html.escape(path_text, quote=True)}'
           data-prediction='{html.escape(node['prediction'], quote=True)}'
           data-samples='{node['samples']}'
           data-gini='{node['gini']:.3f}'
           data-distribution='{html.escape(node['distribution'], quote=True)}'
           onclick='showNodeInfo(this)' onfocus='showNodeInfo(this)'>
          <title>{html.escape(title)}</title>
          <rect x='{rect_x}' y='{rect_y}' width='{node_w}' height='{node_h}' rx='10' ry='10'></rect>
          {svg_lines('노드 ' + str(node['id']), x, rect_y + 22, max_chars=18, max_lines=1, css_class='node-id')}
          {svg_lines('예측: ' + node['prediction'], x, rect_y + 45, max_chars=16, max_lines=2, css_class='node-pred')}
          {svg_lines('샘플 ' + str(node['samples']) + ' / gini ' + f"{node['gini']:.3f}", x, rect_y + 78, max_chars=22, max_lines=1, css_class='node-meta')}
        </g>
        """

    graph_html = f"""
    <div class='graphic-layout'>
      <div>
        <div class='graphic-toolbar'>
          <button onclick='zoomTree(1.2)'>확대</button>
          <button onclick='zoomTree(1 / 1.2)'>축소</button>
          <button onclick='resetTreeView()'>100%</button>
          <span>왼쪽 버튼을 누른 채 드래그하면 트리의 다른 부분으로 이동합니다.</span>
        </div>
      <div id='svg-wrap' class='svg-wrap' data-base-width='{svg_width}' data-base-height='{svg_height}'>
        <div id='svg-scaler' style='width:{svg_width}px;height:{svg_height}px;'>
        <svg id='tree-svg' class='tree-svg' width='{svg_width}' height='{svg_height}' viewBox='0 0 {svg_width} {svg_height}' role='img' aria-label='의사결정트리 그래픽 모드'>
          {edge_svg}
          {node_svg}
        </svg>
        </div>
      </div>
      </div>
      <aside id='node-info' class='node-info'>
        <h2>노드 정보</h2>
        <p>그래픽 트리의 노드를 클릭하거나 마우스를 올리면 이곳에 설명이 표시됩니다.</p>
        {tree_summary_html}
      </aside>
    </div>
    """
    document = f"""
    <!doctype html>
    <html lang='ko'>
    <head>
      <meta charset='utf-8'>
      <title>의사결정트리 인터랙티브 보기</title>
      <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Malgun Gothic', 'Apple SD Gothic Neo', sans-serif; margin: 24px; color: #1f2937; }}
        h1 {{ margin-bottom: 8px; }}
        .summary {{ display: flex; flex-wrap: wrap; gap: 8px; margin: 16px 0 20px; }}
        .summary span, .badge {{ background: #eef2ff; border: 1px solid #c7d2fe; border-radius: 999px; padding: 3px 9px; font-size: 13px; }}
        .help {{ background: #f8fafc; border: 1px solid #cbd5e1; border-radius: 8px; padding: 14px 16px; margin-bottom: 18px; line-height: 1.65; }}
        .tree {{ min-width: 960px; }}
        details.node {{ margin: 8px 0 8px 24px; border-left: 2px solid #cbd5e1; padding-left: 12px; }}
        details.node > summary {{ cursor: pointer; list-style: none; border: 1px solid #d1d5db; border-radius: 8px; padding: 9px 11px; background: #ffffff; display: flex; gap: 8px; align-items: center; flex-wrap: wrap; }}
        details.node > summary::-webkit-details-marker {{ display: none; }}
        details.node.internal > summary {{ border-left: 6px solid #2563eb; }}
        details.node.leaf > summary {{ border-left: 6px solid #16a34a; background: #f0fdf4; }}
        .branch {{ font-weight: 700; color: #0f172a; background: #fef3c7; border: 1px solid #f59e0b; border-radius: 6px; padding: 3px 7px; }}
        .title {{ font-weight: 700; }}
        .body {{ margin: 8px 0 10px 14px; padding: 10px 12px; background: #f8fafc; border: 1px solid #e5e7eb; border-radius: 8px; line-height: 1.55; }}
        .children {{ margin-left: 14px; }}
        .mode-tabs {{ display: flex; gap: 8px; margin: 18px 0 12px; }}
        .mode-tabs button {{ border: 1px solid #94a3b8; background: #fff; border-radius: 7px; padding: 8px 12px; cursor: pointer; font-weight: 700; }}
        .mode-tabs button.active {{ background: #2563eb; color: #fff; border-color: #2563eb; }}
        .mode-panel.hidden {{ display: none; }}
        .graphic-layout {{ display: grid; grid-template-columns: minmax(720px, 1fr) 320px; gap: 16px; align-items: start; }}
        .graphic-toolbar {{ display: flex; flex-wrap: wrap; gap: 8px; align-items: center; margin-bottom: 8px; }}
        .graphic-toolbar button {{ border: 1px solid #94a3b8; background: #fff; border-radius: 7px; padding: 7px 10px; cursor: pointer; font-weight: 700; }}
        .graphic-toolbar span {{ color: #475569; font-size: 13px; }}
        .svg-wrap {{ overflow: auto; border: 1px solid #cbd5e1; border-radius: 10px; background: #f8fafc; height: 680px; cursor: grab; user-select: none; }}
        .svg-wrap.panning {{ cursor: grabbing; }}
        .tree-svg {{ display: block; transform-origin: 0 0; }}
        .edge {{ stroke: #94a3b8; stroke-width: 2; }}
        .edge-label {{ font-size: 12px; fill: #475569; paint-order: stroke; stroke: #f8fafc; stroke-width: 5px; stroke-linejoin: round; }}
        .svg-node {{ cursor: pointer; outline: none; }}
        .svg-node rect {{ fill: #ffffff; stroke: #2563eb; stroke-width: 2; filter: drop-shadow(0 2px 2px rgba(15, 23, 42, 0.12)); }}
        .svg-node.leaf rect {{ fill: #ecfdf5; stroke: #16a34a; }}
        .svg-node:hover rect, .svg-node:focus rect {{ stroke-width: 4; }}
        .node-id {{ font-size: 12px; font-weight: 700; fill: #475569; }}
        .node-pred {{ font-size: 13px; font-weight: 700; fill: #111827; }}
        .node-meta {{ font-size: 11px; fill: #64748b; }}
        .node-info {{ position: sticky; top: 16px; border: 1px solid #cbd5e1; border-radius: 10px; padding: 14px 16px; background: #ffffff; line-height: 1.55; }}
        .node-info h2 {{ margin-top: 0; font-size: 18px; }}
        .path-list {{ margin: 6px 0 0 20px; padding: 0; }}
        .path-list li {{ margin: 4px 0; }}
        .tree-summary {{ margin-top: 16px; padding-top: 14px; border-top: 1px solid #e5e7eb; }}
        .summary-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 8px; }}
        .summary-grid div {{ border: 1px solid #e5e7eb; background: #f8fafc; border-radius: 8px; padding: 8px; }}
        .small-help {{ color: #64748b; font-size: 13px; }}
        .question-table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
        .question-table th, .question-table td {{ border: 1px solid #e5e7eb; padding: 6px; text-align: left; vertical-align: top; }}
        .question-table th {{ background: #f1f5f9; }}
        @media (max-width: 1000px) {{ .graphic-layout {{ grid-template-columns: 1fr; }} .node-info {{ position: static; }} }}
      </style>
    </head>
    <body>
      <h1>의사결정트리 인터랙티브 보기</h1>
      <div class='summary'>
        <span>전체 노드 수: {total_nodes}</span>
        <span>리프 노드 수: {leaf_count}</span>
        <span>최대 레벨: {max_depth}</span>
      </div>
      <div class='help'>
        <b>읽는 방법</b><br>
        각 상자는 하나의 노드입니다. 노드를 클릭하면 하위 분기가 접히거나 펼쳐집니다.
        노란 조건 배지 위에 마우스를 올리면 조건 설명을 볼 수 있습니다.
        범주형 변수는 원래 내부적으로 0과 1로 바뀌지만, 여기서는 <b>읍면동1 = 근화동</b>처럼 사람이 읽는 문장으로 바꾸어 표시했습니다.
        <b>gini</b>는 노드 안에 여러 분류가 얼마나 섞여 있는지를 뜻하며, 0에 가까울수록 한 종류로 잘 모인 상태입니다.
      </div>
      <div class='mode-tabs'>
        <button id='graphic-tab' class='active' onclick="showMode('graphic')">그래픽 모드</button>
        <button id='text-tab' onclick="showMode('text')">텍스트 모드</button>
      </div>
      <section id='graphic-mode' class='mode-panel'>{graph_html}</section>
      <section id='text-mode' class='mode-panel hidden'><div class='tree'>{tree_html}</div></section>
      <script>
        function showMode(mode) {{
          document.getElementById('graphic-mode').classList.toggle('hidden', mode !== 'graphic');
          document.getElementById('text-mode').classList.toggle('hidden', mode !== 'text');
          document.getElementById('graphic-tab').classList.toggle('active', mode === 'graphic');
          document.getElementById('text-tab').classList.toggle('active', mode === 'text');
        }}
        function showNodeInfo(el) {{
          const box = document.getElementById('node-info');
          const branch = el.dataset.branch || '시작';
          const help = el.dataset.help || '';
          const distribution = el.dataset.distribution || '표시할 분포 없음';
          const pathItems = (el.dataset.path || '').split('||').filter(Boolean);
          const pathHtml = pathItems.length
            ? `<ol class='path-list'>${{pathItems.map((item, index) => `<li><b>${{index + 1}}단계:</b> ${{item}}</li>`).join('')}}</ol>`
            : '<p>루트 노드라 이전 분기 조건이 없습니다.</p>';
          box.innerHTML = `
            <h2>노드 ${{el.dataset.node}}</h2>
            <p><b>레벨:</b> ${{el.dataset.level}}</p>
            <p><b>이 노드로 오는 조건:</b><br>${{branch}}</p>
            <p><b>조건 설명:</b><br>${{help}}</p>
            <p><b>루트에서 이 노드까지 온 전체 과정:</b></p>
            ${{pathHtml}}
            <p><b>예측값:</b> ${{el.dataset.prediction}}</p>
            <p><b>샘플 수:</b> ${{el.dataset.samples}}</p>
            <p><b>gini:</b> ${{el.dataset.gini}}<br><small>0에 가까울수록 한 종류의 값이 많이 모인 노드입니다.</small></p>
            <p><b>분류 분포:</b><br>${{distribution}}</p>
            {tree_summary_html}
          `;
        }}
        let treeScale = 1;
        function applyTreeScale() {{
          const wrap = document.getElementById('svg-wrap');
          const scaler = document.getElementById('svg-scaler');
          const svg = document.getElementById('tree-svg');
          const baseWidth = Number(wrap.dataset.baseWidth);
          const baseHeight = Number(wrap.dataset.baseHeight);
          scaler.style.width = `${{baseWidth * treeScale}}px`;
          scaler.style.height = `${{baseHeight * treeScale}}px`;
          svg.style.transform = `scale(${{treeScale}})`;
        }}
        function zoomTree(factor) {{
          treeScale = Math.min(3, Math.max(0.35, treeScale * factor));
          applyTreeScale();
        }}
        function resetTreeView() {{
          treeScale = 1;
          applyTreeScale();
        }}
        const wrap = document.getElementById('svg-wrap');
        let isPanning = false;
        let panStartX = 0;
        let panStartY = 0;
        let startScrollLeft = 0;
        let startScrollTop = 0;
        wrap.addEventListener('mousedown', (event) => {{
          if (event.button !== 0) return;
          isPanning = true;
          wrap.classList.add('panning');
          panStartX = event.clientX;
          panStartY = event.clientY;
          startScrollLeft = wrap.scrollLeft;
          startScrollTop = wrap.scrollTop;
        }});
        window.addEventListener('mousemove', (event) => {{
          if (!isPanning) return;
          event.preventDefault();
          wrap.scrollLeft = startScrollLeft - (event.clientX - panStartX);
          wrap.scrollTop = startScrollTop - (event.clientY - panStartY);
        }});
        window.addEventListener('mouseup', () => {{
          isPanning = false;
          wrap.classList.remove('panning');
        }});
        applyTreeScale();
      </script>
    </body>
    </html>
    """
    Path(output_path).write_text(document, encoding="utf-8")
    return output_path


def leaf_rules(model, feature_names):
    tree = model.tree_
    rows = []

    def walk(node_id, conditions):
        left = tree.children_left[node_id]
        right = tree.children_right[node_id]

        if left == right:
            values = tree.value[node_id][0]
            pred = model.classes_[int(np.argmax(values))]
            rows.append({
                "리프노드": node_id,
                "조건": " AND ".join(conditions) if conditions else "전체",
                "예측값": pred,
                "샘플 수": int(tree.n_node_samples[node_id]),
                "불순도(gini)": round(float(tree.impurity[node_id]), 4),
            })
            return

        feature_name = feature_names[tree.feature[node_id]]
        threshold = tree.threshold[node_id]
        left_text, _ = readable_condition(feature_name, threshold, "left")
        right_text, _ = readable_condition(feature_name, threshold, "right")
        walk(left, conditions + [left_text])
        walk(right, conditions + [right_text])

    walk(0, [])
    return pd.DataFrame(rows)


def save_and_show_tree(model, X, base_name, plot_depth=None):
    feature_names = [str(col) for col in X.columns]
    class_names = [str(label) for label in model.classes_]

    width = min(max(14, len(feature_names) * 0.25), 34)
    height = min(max(8, model.get_depth() * 2.2 + 3), 30)
    fig, ax = plt.subplots(figsize=(width, height))

    plot_tree(
        model,
        feature_names=feature_names,
        class_names=class_names,
        filled=True,
        rounded=True,
        fontsize=9,
        max_depth=plot_depth,
        ax=ax,
    )
    ax.set_title("의사결정트리 분류 결과", fontsize=14, pad=16)
    plt.tight_layout()

    png_path = f"{base_name}.png"
    pdf_path = f"{base_name}.pdf"
    rules_csv_path = f"{base_name}_rules.csv"
    rules_html_path = f"{base_name}_rules.html"
    interactive_html_path = f"{base_name}_interactive_tree.html"

    fig.savefig(png_path, dpi=220, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    rules_df = leaf_rules(model, feature_names)
    rules_df.to_csv(rules_csv_path, index=False, encoding="utf-8-sig")
    Path(rules_html_path).write_text(rules_df.to_html(index=False), encoding="utf-8")
    make_interactive_tree_html(model, feature_names, interactive_html_path)

    return png_path, pdf_path, rules_csv_path, rules_html_path, interactive_html_path, rules_df

print("의사 결정 나무를 만들기 위한 작업이 완료되었습니다.")



## 4. 변수(속성) 선택 후 의사결정트리 생성

In [ ]:
file_name = "춘천 닭갈비(2021) 상위 15개 점포 정보"

# @title
target_dropdown = widgets.Dropdown(options=list(df.columns), description="종속변수")
feature_box = widgets.VBox()
advanced_checkbox = widgets.Checkbox(value=False, description="고급 설정 사용")
max_depth_slider = widgets.IntSlider(value=4, min=1, max=12, step=1, description="최대깊이", continuous_update=False)
min_leaf_slider = widgets.IntSlider(value=1, min=1, max=20, step=1, description="리프최소", continuous_update=False)
limit_plot_checkbox = widgets.Checkbox(value=False, description="그림만 4단계까지 표시")
advanced_box = widgets.VBox([max_depth_slider, min_leaf_slider, limit_plot_checkbox])
run_button = widgets.Button(description="의사결정트리 생성", button_style="success")
tree_output = widgets.Output()
feature_checkboxes = []


def refresh_features(change=None):
    global feature_checkboxes
    target = target_dropdown.value
    feature_checkboxes = []

    for col in df.columns:
        if col == target:
            continue
        checkbox = widgets.Checkbox(value=False, description=str(col), indent=False)
        feature_checkboxes.append(checkbox)

    feature_box.children = feature_checkboxes


def refresh_advanced_box(change=None):
    advanced_box.layout.display = "" if advanced_checkbox.value else "none"


def selected_features():
    return [checkbox.description for checkbox in feature_checkboxes if checkbox.value]


def run_decision_tree(_):
    with tree_output:
        clear_output()
        target_col = target_dropdown.value
        feature_cols = selected_features()

        if not feature_cols:
            display(HTML("<b style='color:#b00020'>독립변수를 1개 이상 체크해 주세요.</b>"))
            return

        max_depth = max_depth_slider.value if advanced_checkbox.value else None
        min_samples_leaf = min_leaf_slider.value if advanced_checkbox.value else 1
        plot_depth = 4 if advanced_checkbox.value and limit_plot_checkbox.value else None

        try:
            model, X, y, metrics, messages, warning_messages = train_tree(
                df,
                feature_cols,
                target_col,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
            )
        except Exception as exc:
            display(HTML(f"<b style='color:#b00020'>트리 생성 실패:</b><pre>{exc}</pre>"))
            return

        display(HTML("<h3>선택한 변수</h3>"))
        display(pd.DataFrame({"구분": ["종속변수", "독립변수"], "값": [target_col, ", ".join(feature_cols)]}))

        display(HTML("<h3>전처리 설명</h3>"))
        for msg in messages:
            display(HTML(f"<div>• {msg}</div>"))

        if warning_messages:
            display(HTML("<h3 style='color:#9a5b00'>주의할 점</h3>"))
            for msg in warning_messages:
                display(HTML(f"<div style='color:#9a5b00'>• {msg}</div>"))

        display(HTML("<h3>트리 요약</h3>"))
        display(pd.DataFrame([metrics]).T.rename(columns={0: "값"}))

        base_name = "decision_tree_" + safe_stem(file_name)
        png_path, pdf_path, rules_csv_path, rules_html_path, interactive_html_path, rules_df = save_and_show_tree(model, X, base_name, plot_depth)

        display(HTML("<h3>저장된 파일</h3>"))
        display(pd.DataFrame([
            {"파일 종류": "트리 이미지 PNG", "파일명": png_path},
            {"파일 종류": "트리 문서 PDF", "파일명": pdf_path},
            {"파일 종류": "인터랙티브 트리 HTML", "파일명": interactive_html_path},
            {"파일 종류": "리프 규칙표 CSV", "파일명": rules_csv_path},
            {"파일 종류": "리프 규칙표 HTML", "파일명": rules_html_path},
        ]))

        display(HTML("<h4>리프 노드 규칙표 미리보기</h4>"))
        display(rules_df.head(30))


target_dropdown.observe(refresh_features, names="value")
advanced_checkbox.observe(refresh_advanced_box, names="value")
run_button.on_click(run_decision_tree)
refresh_features()
refresh_advanced_box()

display(widgets.VBox([
    target_dropdown,
    widgets.HTML("<b>독립변수</b>"),
    feature_box,
    advanced_checkbox,
    advanced_box,
    run_button,
    tree_output,
]))
